# DeepLabV3+ v5 — Seismic Facies Segmentation (Final)
**Dataset:** F3 Block Netherlands — Alaudah et al. (2019)  
**Model:** DeepLabV3+ (EfficientNet-B4 encoder, ImageNet pretrained, ASPP)  

## v4 → v5 Changes
| Change | v4 | v5 |
|---|---|---|
| Encoder | ResNet-50 | EfficientNet-B4 |
| ASPP rates | (6,12,18) | (12,24,36) |
| Image size | 256×256 | 320×320 |
| S4 crossline oversampling | ❌ | ✅ WeightedRandomSampler |
| Mixup augmentation | ❌ | ✅ alpha=0.2 |
| Label smoothing | ❌ | ✅ eps=0.1 |
| All plots | Turkish | English |

## 0. Setup

In [ ]:
import subprocess, sys
pkgs = ["matplotlib", "scikit-learn", "albumentations", "segmentation-models-pytorch"]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + pkgs)
print("Installation complete.")

In [ ]:
import os, sys, random, json, time, zipfile
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset, WeightedRandomSampler
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import segmentation_models_pytorch as smp
import albumentations as A

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device("cpu")
    print("WARNING: No GPU found, using CPU.")
print(f"Device: {device} | PyTorch: {torch.__version__} | SMP: {smp.__version__}")

NOTEBOOK_DIR = Path(".").resolve()
DATA_DIR     = NOTEBOOK_DIR / "data"
CHECKPOINTS  = NOTEBOOK_DIR / "checkpoints_v5"
FIGURES_DIR  = NOTEBOOK_DIR / "results_v5" / "figures"
METRICS_DIR  = NOTEBOOK_DIR / "results_v5" / "metrics"

for d in [DATA_DIR, CHECKPOINTS, FIGURES_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project : {NOTEBOOK_DIR}")
print(f"Data    : {DATA_DIR}")

## 1. Data Loading

In [ ]:
TRAIN_SEISMIC = DATA_DIR / "train" / "train_seismic.npy"

if not TRAIN_SEISMIC.exists():
    zip_path = DATA_DIR / "data.zip"
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print("Downloading data (~1 GB)...")

    def _progress(block, block_size, total):
        downloaded = block * block_size
        if total > 0:
            pct = min(100, downloaded * 100 / total)
            print(f"\r  {pct:.1f}%  ({downloaded/1e6:.0f} MB)", end="", flush=True)

    urlretrieve("https://zenodo.org/record/3755060/files/data.zip",
                zip_path, reporthook=_progress)
    print("\nExtracting...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(DATA_DIR)
    zip_path.unlink()
    print("Done.")
else:
    print("Data already exists.")

print("Loading NPY files...")
train_seismic = np.load(DATA_DIR / "train"     / "train_seismic.npy")
train_labels  = np.load(DATA_DIR / "train"     / "train_labels.npy")
test1_seismic = np.load(DATA_DIR / "test_once" / "test1_seismic.npy")
test1_labels  = np.load(DATA_DIR / "test_once" / "test1_labels.npy")
test2_seismic = np.load(DATA_DIR / "test_once" / "test2_seismic.npy")
test2_labels  = np.load(DATA_DIR / "test_once" / "test2_labels.npy")

print(f"Train : {train_seismic.shape}  dtype={train_seismic.dtype}")
print(f"Test1 : {test1_seismic.shape}  (inline)")
print(f"Test2 : {test2_seismic.shape}  (crossline)")

INLINE_DIM    = 0
CROSSLINE_DIM = 1

## 2. EDA

In [ ]:
CLASS_NAMES   = ["Upper NS", "Lower NS", "Rijnland", "Scruff", "Zechstein", "Under Zech"]
NUM_CLASSES   = 6
FACIES_COLORS = ["#3288bd", "#66c2a5", "#abdda4", "#e6f598", "#fdae61", "#f46d43"]
cmap_facies   = mcolors.ListedColormap(FACIES_COLORS)

counts = np.bincount(train_labels.flatten(), minlength=NUM_CLASSES)
total  = counts.sum()

print("Class distribution (Train):")
for i, (name, cnt) in enumerate(zip(CLASS_NAMES, counts)):
    print(f"  S{i} {name:22s}: {cnt:>12,}  ({cnt/total*100:.2f}%)")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
bars = axes[0].bar(range(NUM_CLASSES), counts / 1e6, color=FACIES_COLORS, edgecolor="k", lw=0.5)
axes[0].set_xticks(range(NUM_CLASSES))
axes[0].set_xticklabels([f"S{j}\n{CLASS_NAMES[j]}" for j in range(NUM_CLASSES)], fontsize=8)
axes[0].set_ylabel("Pixels (Millions)")
axes[0].set_title("Class Distribution (Train)")
for bar, cnt in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f"{cnt/total*100:.1f}%", ha="center", fontsize=8)

idx = 200
axes[1].imshow(train_seismic[idx].T, cmap="seismic", aspect="auto", vmin=-1, vmax=1)
axes[1].set_title(f"Sample Seismic — Inline #{idx}")
axes[1].set_xlabel("Crossline"); axes[1].set_ylabel("Depth")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "eda.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {FIGURES_DIR}/eda.png")

## 3. Train/Val Split and Class Weights

In [ ]:
all_inline_idx    = np.arange(train_seismic.shape[INLINE_DIM])
train_inline_idx, val_inline_idx = train_test_split(
    all_inline_idx, test_size=0.2, random_state=SEED)

all_crossline_idx = np.arange(train_seismic.shape[CROSSLINE_DIM])
train_cl_idx, val_cl_idx = train_test_split(
    all_crossline_idx, test_size=0.2, random_state=SEED)

print(f"Inline    — Train: {len(train_inline_idx)}  Val: {len(val_inline_idx)}")
print(f"Crossline — Train: {len(train_cl_idx)}  Val: {len(val_cl_idx)}")
print(f"Test1: {test1_seismic.shape[0]} inline  |  Test2: {test2_seismic.shape[1]} crossline")

# Effective Number of Samples weights
train_counts  = np.bincount(train_labels[train_inline_idx].flatten(),
                            minlength=NUM_CLASSES).astype(float)
beta = 0.9999
en   = (1 - beta ** train_counts) / (1 - beta)
class_weights  = 1.0 / en
class_weights /= class_weights.sum() / NUM_CLASSES
weights_tensor = torch.FloatTensor(class_weights).to(device)

print("\nClass weights (Effective Number):")
for i, (name, w) in enumerate(zip(CLASS_NAMES, class_weights)):
    print(f"  S{i} {name:22s}: {w:.4f}")

## 4. Dataset and DataLoader
**FIX: S4 (Zechstein) Crossline Oversampling with WeightedRandomSampler**

In [ ]:
# v5: larger image size for more spatial detail
IMG_SIZE   = (320, 320)
BATCH_SIZE = 6   # reduced for 320x320 + EfficientNet-B4 on 12GB VRAM

train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
    A.GaussNoise(std_range=(0.001, 0.02), p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.ElasticTransform(alpha=30, sigma=5, p=0.3),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=5,
                       border_mode=0, p=0.3),
    A.CoarseDropout(max_holes=8, max_height=20, max_width=20,
                    min_holes=1, fill_value=0, p=0.2),
])


class F3InlineDataset(Dataset):
    """Inline (axis=0) slices"""
    def __init__(self, seismic, labels, indices, img_size=IMG_SIZE,
                 transform=None, augment_polarity=False):
        mean = seismic.mean(); std = seismic.std() + 1e-8
        self.seismic  = ((seismic - mean) / std).astype(np.float32)
        self.labels   = labels.astype(np.int64)
        self.indices  = indices
        self.img_size = img_size
        self.transform = transform
        self.augment_polarity = augment_polarity

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        i    = self.indices[idx]
        img  = self.seismic[i].astype(np.float32)
        mask = self.labels[i].astype(np.int64)
        if self.augment_polarity and random.random() > 0.5:
            img = -img
        if self.transform is not None:
            aug  = self.transform(image=img, mask=mask.astype(np.uint8))
            img  = aug["image"]
            mask = aug["mask"].astype(np.int64)
        img_t  = torch.from_numpy(img).unsqueeze(0)
        mask_t = torch.from_numpy(mask)
        img_t  = F.interpolate(img_t.unsqueeze(0), size=self.img_size,
                               mode="bilinear", align_corners=False).squeeze(0)
        mask_t = F.interpolate(mask_t.float().unsqueeze(0).unsqueeze(0),
                               size=self.img_size, mode="nearest").squeeze(0).squeeze(0).long()
        return img_t, mask_t


class F3CrosslineDataset(Dataset):
    """Crossline (axis=1) slices"""
    def __init__(self, seismic, labels, indices, img_size=IMG_SIZE,
                 transform=None, augment_polarity=False):
        mean = seismic.mean(); std = seismic.std() + 1e-8
        self.seismic  = ((seismic - mean) / std).astype(np.float32)
        self.labels   = labels.astype(np.int64)
        self.indices  = indices
        self.img_size = img_size
        self.transform = transform
        self.augment_polarity = augment_polarity

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        i    = self.indices[idx]
        img  = self.seismic[:, i, :].astype(np.float32)
        mask = self.labels[:, i, :].astype(np.int64)
        if self.augment_polarity and random.random() > 0.5:
            img = -img
        if self.transform is not None:
            aug  = self.transform(image=img, mask=mask.astype(np.uint8))
            img  = aug["image"]
            mask = aug["mask"].astype(np.int64)
        img_t  = torch.from_numpy(img).unsqueeze(0)
        mask_t = torch.from_numpy(mask)
        img_t  = F.interpolate(img_t.unsqueeze(0), size=self.img_size,
                               mode="bilinear", align_corners=False).squeeze(0)
        mask_t = F.interpolate(mask_t.float().unsqueeze(0).unsqueeze(0),
                               size=self.img_size, mode="nearest").squeeze(0).squeeze(0).long()
        return img_t, mask_t


# ---------------------------------------------------------
# FIX: S4 (Zechstein) Crossline Oversampling
# S4 has IoU=23.9% on Test2 but 81.8% on Test1.
# Root cause: Zechstein looks geometrically different
# along the crossline direction. We oversample crossline
# slices that contain significant S4 pixels (>=5% of slice).
# ---------------------------------------------------------
S4_CLASS       = 4   # Zechstein index
S4_THRESHOLD   = 0.05  # at least 5% S4 pixels in slice
S4_OVERSAMPLE  = 3     # repeat S4-rich crossline slices 3x

def build_weighted_sampler(inline_ds, crossline_ds, s4_cl_indices,
                           train_cl_idx, s4_oversample=S4_OVERSAMPLE):
    """
    Build WeightedRandomSampler for ConcatDataset.
    Crossline slices with high S4 content get higher weight.
    """
    n_inline = len(inline_ds)
    n_cl     = len(crossline_ds)
    n_total  = n_inline + n_cl

    weights = np.ones(n_total, dtype=np.float32)

    # Create a set of high-S4 crossline positions in the concat index
    s4_rich_set = set(s4_cl_indices)
    for local_i, cl_idx in enumerate(train_cl_idx):
        if cl_idx in s4_rich_set:
            concat_i = n_inline + local_i   # offset by inline dataset size
            weights[concat_i] = float(s4_oversample)

    sampler = WeightedRandomSampler(
        weights=torch.from_numpy(weights),
        num_samples=n_total,
        replacement=True
    )
    return sampler


# Find S4-rich crossline slices in TRAINING set
print("Finding S4-rich crossline slices...")
s4_rich_cl_indices = []
for cl_idx in train_cl_idx:
    slice_labels = train_labels[:, cl_idx, :]
    s4_frac = (slice_labels == S4_CLASS).sum() / slice_labels.size
    if s4_frac >= S4_THRESHOLD:
        s4_rich_cl_indices.append(cl_idx)

print(f"  Total crossline train slices  : {len(train_cl_idx)}")
print(f"  S4-rich crossline slices (>={S4_THRESHOLD*100:.0f}%): {len(s4_rich_cl_indices)}")
print(f"  Oversampling factor           : {S4_OVERSAMPLE}x")

# Build datasets
NUM_WORKERS = 0 if sys.platform == "win32" else 2
pin = device.type == "cuda"

train_inline_ds = F3InlineDataset(train_seismic, train_labels, train_inline_idx,
                                  transform=train_transform, augment_polarity=True)
train_cl_ds     = F3CrosslineDataset(train_seismic, train_labels, train_cl_idx,
                                     transform=train_transform, augment_polarity=True)
train_ds_combined = ConcatDataset([train_inline_ds, train_cl_ds])

val_ds   = F3InlineDataset(train_seismic, train_labels, val_inline_idx)
test1_ds = F3InlineDataset(test1_seismic, test1_labels, np.arange(test1_seismic.shape[0]))
test2_ds = F3CrosslineDataset(test2_seismic, test2_labels, np.arange(test2_seismic.shape[1]))

# Weighted sampler for S4 oversampling
sampler = build_weighted_sampler(train_inline_ds, train_cl_ds,
                                 s4_rich_cl_indices, train_cl_idx)

train_loader = DataLoader(train_ds_combined, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=pin)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=pin)
test1_loader = DataLoader(test1_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test2_loader = DataLoader(test2_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"\nTrain combined : {len(train_ds_combined)} ({len(train_inline_ds)} inline + {len(train_cl_ds)} crossline)")
print(f"Val            : {len(val_ds)}")
print(f"Test1 (inline) : {len(test1_ds)}")
print(f"Test2 (cross)  : {len(test2_ds)}")
print(f"Image size     : {IMG_SIZE}")
imgs, masks = next(iter(train_loader))
print(f"Batch — image: {imgs.shape}  mask: {masks.shape}")

## 5. Model — DeepLabV3+ (EfficientNet-B4)
**FIX: EfficientNet-B4 encoder + wider ASPP rates (12,24,36)**

In [ ]:
# v5: EfficientNet-B4 — better accuracy/VRAM ratio than ResNet-50
# ASPP rates (12,24,36) — larger receptive field for geological layers
model = smp.DeepLabV3Plus(
    encoder_name="efficientnet-b4",
    encoder_weights="imagenet",
    in_channels=1,
    classes=NUM_CLASSES,
    activation=None,
    encoder_output_stride=16,
    decoder_atrous_rates=(12, 24, 36),   # wider than v4's (6,12,18)
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {n_params:,}")
print(f"Encoder: EfficientNet-B4 (ImageNet pretrained)")
print(f"ASPP rates: (12, 24, 36)")

with torch.no_grad():
    dummy = torch.randn(2, 1, *IMG_SIZE).to(device)
    out   = model(dummy)
    print(f"Output shape: {out.shape}  (expected: [2, 6, {IMG_SIZE[0]}, {IMG_SIZE[1]}])")
del dummy, out
if device.type == 'cuda':
    torch.cuda.empty_cache()
print("Model ready.")

## 6. Loss, Optimizer, Scheduler
**FIX: Label Smoothing CrossEntropy + Mixup augmentation**

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, n_classes=NUM_CLASSES, smooth=1.0):
        super().__init__()
        self.n_classes = n_classes
        self.smooth    = smooth

    def forward(self, pred, target):
        pred_soft = torch.softmax(pred, dim=1)
        target_oh = F.one_hot(target, self.n_classes).permute(0, 3, 1, 2).float()
        dice = 0.0
        for c in range(self.n_classes):
            p = pred_soft[:, c]; t = target_oh[:, c]
            inter = (p * t).sum()
            dice += (2 * inter + self.smooth) / (p.sum() + t.sum() + self.smooth)
        return 1 - dice / self.n_classes


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight

    def forward(self, pred, target):
        ce_loss = F.cross_entropy(pred, target, weight=self.weight, reduction='none')
        pt      = torch.exp(-ce_loss)
        focal   = ((1 - pt) ** self.gamma) * ce_loss
        return focal.mean()


class TripleLoss(nn.Module):
    """
    0.4 * LabelSmoothingCE + 0.3 * Dice + 0.3 * Focal
    v5 change: label_smoothing=0.1 added to CE
      -> prevents overconfidence on majority classes
      -> helps model stay calibrated for S4 crossline
    """
    def __init__(self, weights, label_smoothing=0.1):
        super().__init__()
        self.ce    = nn.CrossEntropyLoss(weight=weights, label_smoothing=label_smoothing)
        self.dice  = DiceLoss()
        self.focal = FocalLoss(gamma=2.0, weight=weights)

    def forward(self, pred, target):
        return (0.4 * self.ce(pred, target)
              + 0.3 * self.dice(pred, target)
              + 0.3 * self.focal(pred, target))


def mixup_data(x, y, alpha=0.2):
    """
    Mixup augmentation — v5 addition.
    Blends two training samples in both input and label space.
    Helps the model generalize across the inline/crossline boundary.
    Returns mixed_x, y_a, y_b, lambda
    """
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


NUM_EPOCHS = 100
MIXUP_ALPHA = 0.2   # beta distribution parameter

criterion = TripleLoss(weights_tensor, label_smoothing=0.1)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4,
    betas=(0.9, 0.999)
)

# CosineAnnealingWarmRestarts — same as v4, proven effective
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=25, T_mult=1, eta_min=1e-6
)

print("Loss       : 0.4*LabelSmoothing(0.1)CE + 0.3*Dice + 0.3*Focal(gamma=2)")
print("Optimizer  : AdamW  lr=1e-4  wd=1e-4")
print("Scheduler  : CosineAnnealingWarmRestarts  T_0=25")
print(f"Epochs     : {NUM_EPOCHS}")
print(f"Mixup      : alpha={MIXUP_ALPHA}")
print("GradClip   : max_norm=1.0")

## 7. Metric Functions

In [ ]:
def compute_metrics(preds_all, targets_all, n=NUM_CLASSES):
    res = {}
    res['PA'] = float((preds_all == targets_all).sum() / len(targets_all))
    cm = confusion_matrix(targets_all, preds_all, labels=list(range(n)))

    row_sums  = cm.sum(axis=1).astype(float)
    class_acc = np.where(row_sums > 0, cm.diagonal() / row_sums, 0.0)
    res['MCA'] = float(class_acc.mean())
    res['per_class_acc'] = class_acc.tolist()

    ious = []
    for c in range(n):
        inter = cm[c, c]; union = cm[c,:].sum() + cm[:,c].sum() - inter
        ious.append(float(inter / (union + 1e-8)))
    res['mIoU'] = float(np.mean(ious))
    res['per_class_iou'] = ious

    dices = []
    for c in range(n):
        tp = cm[c,c]; fp = cm[:,c].sum()-tp; fn = cm[c,:].sum()-tp
        dices.append(float(2*tp / (2*tp + fp + fn + 1e-8)))
    res['mean_dice'] = float(np.mean(dices))
    res['per_class_dice'] = dices
    res['confusion_matrix'] = cm.tolist()
    return res


def print_metrics(metrics, title='Model'):
    print(f'\n{"="*60}')
    print(f' {title}')
    print(f'{"="*60}')
    print(f'  Pixel Accuracy (PA) : {metrics["PA"]*100:.2f}%')
    print(f'  Mean Class Acc (MCA): {metrics["MCA"]*100:.2f}%')
    print(f'  Mean IoU (mIoU)     : {metrics["mIoU"]*100:.2f}%')
    print(f'  Mean Dice           : {metrics["mean_dice"]*100:.2f}%')
    print('  Per-class IoU:')
    for i, name in enumerate(CLASS_NAMES):
        iou  = metrics['per_class_iou'][i]
        dice = metrics['per_class_dice'][i]
        flag = " <-- needs improvement" if iou < 0.3 else ""
        print(f"    S{i} {name:20s}: IoU={iou:.4f}  Dice={dice:.4f}{flag}")


print("Metric functions ready.")

## 8. Training (100 Epochs) with Mixup

In [ ]:
CHECKPOINT_PATH = CHECKPOINTS / "deeplabv3plus_v5_checkpoint.pth"
BEST_MODEL_PATH = CHECKPOINTS / "deeplabv3plus_v5_best.pth"
MAX_GRAD_NORM   = 1.0

use_amp = device.type == "cuda"
scaler  = torch.amp.GradScaler("cuda") if use_amp else None

start_epoch   = 0
best_val_miou = 0.0
history = {'train_loss': [], 'val_loss': [], 'val_miou': [], 'val_dice': [], 'lr': []}

if CHECKPOINT_PATH.exists():
    print("Checkpoint found, resuming...")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    start_epoch   = ckpt['epoch'] + 1
    best_val_miou = ckpt['best_val_miou']
    history       = ckpt['history']
    if use_amp and 'scaler_state' in ckpt:
        scaler.load_state_dict(ckpt['scaler_state'])
    print(f"Resuming from epoch {start_epoch}/{NUM_EPOCHS} — best mIoU so far: {best_val_miou:.4f}")
else:
    print("Starting from scratch...")

print(f"\nDevice: {device}  |  AMP: {use_amp}  |  Remaining epochs: {NUM_EPOCHS - start_epoch}")
print(f"Train batches: {len(train_loader)}  Val batches: {len(val_loader)}")
print("=" * 85)

for epoch in range(start_epoch, NUM_EPOCHS):
    t0 = time.time()

    # --- TRAIN with Mixup ---
    model.train()
    train_loss = 0.0
    for imgs, masks in train_loader:
        imgs  = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        # Mixup — applied 50% of the time to avoid full mixup overhead
        use_mixup = random.random() < 0.5
        if use_mixup:
            imgs_mix, masks_a, masks_b, lam = mixup_data(imgs, masks, alpha=MIXUP_ALPHA)
        else:
            imgs_mix = imgs

        optimizer.zero_grad()

        if use_amp:
            with torch.amp.autocast("cuda"):
                preds = model(imgs_mix)
                if use_mixup:
                    loss = mixup_criterion(criterion, preds, masks_a, masks_b, lam)
                else:
                    loss = criterion(preds, masks)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
        else:
            preds = model(imgs_mix)
            if use_mixup:
                loss = mixup_criterion(criterion, preds, masks_a, masks_b, lam)
            else:
                loss = criterion(preds, masks)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()

        train_loss += loss.item()
    train_loss /= len(train_loader)

    # --- VALIDATION (no mixup) ---
    model.eval()
    val_loss = 0.0
    all_p, all_t = [], []
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device, non_blocking=True), masks.to(device, non_blocking=True)
            if use_amp:
                with torch.amp.autocast("cuda"):
                    preds = model(imgs)
            else:
                preds = model(imgs)
            val_loss += criterion(preds, masks).item()
            all_p.append(preds.argmax(1).cpu().numpy().flatten())
            all_t.append(masks.cpu().numpy().flatten())

    val_loss   /= len(val_loader)
    val_metrics = compute_metrics(np.concatenate(all_p), np.concatenate(all_t))

    scheduler.step()
    lr = optimizer.param_groups[0]['lr']

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_miou'].append(val_metrics['mIoU'])
    history['val_dice'].append(val_metrics['mean_dice'])
    history['lr'].append(lr)

    marker = ""
    if val_metrics['mIoU'] > best_val_miou:
        best_val_miou = val_metrics['mIoU']
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        marker = "  *** BEST ***"

    ckpt_data = {
        'epoch': epoch, 'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'best_val_miou': best_val_miou, 'history': history,
    }
    if use_amp:
        ckpt_data['scaler_state'] = scaler.state_dict()
    torch.save(ckpt_data, CHECKPOINT_PATH)

    elapsed = time.time() - t0
    print(f"Ep {epoch+1:03d}/{NUM_EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f} "
          f"| mIoU: {val_metrics['mIoU']:.4f} | Dice: {val_metrics['mean_dice']:.4f} "
          f"| lr: {lr:.2e} | {elapsed:.0f}s{marker}")

print(f"\nTraining complete! Best val mIoU: {best_val_miou:.4f}")

## 9. Test Evaluation with TTA

In [ ]:
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device, weights_only=True))
model.eval()
print(f"Best model loaded: {BEST_MODEL_PATH}")


def predict_with_tta(imgs, model, device, use_amp):
    """Test-Time Augmentation: average of original + horizontal flip"""
    imgs = imgs.to(device, non_blocking=True)
    if use_amp:
        with torch.amp.autocast("cuda"):
            logits_orig = model(imgs)
            logits_flip = model(torch.flip(imgs, dims=[3]))
    else:
        logits_orig = model(imgs)
        logits_flip = model(torch.flip(imgs, dims=[3]))
    prob_orig = torch.softmax(logits_orig, dim=1)
    prob_flip = torch.softmax(torch.flip(logits_flip, dims=[3]), dim=1)
    return ((prob_orig + prob_flip) / 2.0).argmax(dim=1)


def run_eval(loader):
    all_p, all_t = [], []
    with torch.no_grad():
        for imgs, masks in loader:
            preds = predict_with_tta(imgs, model, device, use_amp)
            all_p.append(preds.cpu().numpy().flatten())
            all_t.append(masks.numpy().flatten())
    return np.concatenate(all_p), np.concatenate(all_t)


print("Evaluating Test1 (Inline)...")
p1, t1   = run_eval(test1_loader); metrics1   = compute_metrics(p1, t1)
print("Evaluating Test2 (Crossline)...")
p2, t2   = run_eval(test2_loader); metrics2   = compute_metrics(p2, t2)
metrics_all = compute_metrics(np.concatenate([p1,p2]), np.concatenate([t1,t2]))

print_metrics(metrics1,    "Test1 — Inline")
print_metrics(metrics2,    "Test2 — Crossline (Generalization)")
print_metrics(metrics_all, "Combined Test1 + Test2")

results = {
    "model": "DeepLabV3+ v5 (EfficientNet-B4)",
    "test1": metrics1, "test2": metrics2,
    "combined": metrics_all, "history": history
}
with open(METRICS_DIR / "deeplabv3plus_v5_metrics.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"\nMetrics saved: {METRICS_DIR}/deeplabv3plus_v5_metrics.json")

## 10. Version Comparison (v3 / v4 / v5)

In [ ]:
v3 = {"test1_miou": 0.6446, "test2_miou": 0.2710, "combined_miou": 0.4012,
      "combined_dice": 0.5453, "combined_pa": 0.6953}
v4 = {"test1_miou": 0.7639, "test2_miou": 0.6838, "combined_miou": 0.7679,
      "combined_dice": 0.8594, "combined_pa": 0.9351}
v5 = {"test1_miou": metrics1['mIoU'], "test2_miou": metrics2['mIoU'],
      "combined_miou": metrics_all['mIoU'], "combined_dice": metrics_all['mean_dice'],
      "combined_pa": metrics_all['PA']}

print("=" * 70)
print(" Version Comparison")
print("=" * 70)
fmt = "{:30s} | {:>8s} | {:>8s} | {:>8s}"
print(fmt.format("Metric", "v3", "v4", "v5"))
print("-" * 70)
for key, label in [
    ("test1_miou",    "Test1 mIoU (inline)"),
    ("test2_miou",    "Test2 mIoU (crossline)"),
    ("combined_miou", "Combined mIoU"),
    ("combined_dice", "Combined Dice"),
    ("combined_pa",   "Combined PA"),
]:
    v3v = v3[key]; v4v = v4[key]; v5v = v5[key]
    best_marker = " <--" if v5v >= max(v3v, v4v, v5v) else ""
    print(fmt.format(label, f"{v3v:.4f}", f"{v4v:.4f}", f"{v5v:.4f}") + best_marker)
print("=" * 70)

print("\nPer-class IoU (Combined) — v3 vs v4 vs v5:")
v3_cls = [0.5817, 0.3460, 0.6073, 0.2802, 0.5087, 0.0835]
v4_cls = [0.9510, 0.8182, 0.9318, 0.6070, 0.7760, 0.5235]
v5_cls = metrics_all['per_class_iou']
for i, name in enumerate(CLASS_NAMES):
    print(f"  S{i} {name:20s}: v3={v3_cls[i]:.4f}  v4={v4_cls[i]:.4f}  v5={v5_cls[i]:.4f}")

## 11. Training Curves

In [ ]:
epochs_x = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(epochs_x, history['train_loss'], label='Train', color='steelblue')
axes[0].plot(epochs_x, history['val_loss'],   label='Val',   color='coral')
axes[0].set_title('Loss Curves'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')

axes[1].plot(epochs_x, history['val_miou'], label='mIoU',  color='green')
axes[1].plot(epochs_x, history['val_dice'], label='Dice',  color='purple')
axes[1].axhline(0.9, color='gray', ls='--', alpha=0.5, label='0.90 target')
axes[1].set_title('Validation Metrics'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Score')

axes[2].semilogy(epochs_x, history['lr'], color='darkorange')
axes[2].set_title('Learning Rate (WarmRestarts)'); axes[2].grid(alpha=0.3)
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('LR')

plt.suptitle("DeepLabV3+ v5 — Training History", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "training_curves_v5.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {FIGURES_DIR}/training_curves_v5.png")

## 12. Segmentation Comparison — Test1 and Test2

In [ ]:
model.eval()
n_samples = 4

for test_ds, test_name, metrics in [
        (test1_ds, "Test1 (Inline)",     metrics1),
        (test2_ds, "Test2 (Crossline)",  metrics2)]:

    sample_idx = np.linspace(0, len(test_ds)-1, n_samples, dtype=int)
    fig, axes  = plt.subplots(n_samples, 3, figsize=(14, 4 * n_samples))

    with torch.no_grad():
        for row, idx in enumerate(sample_idx):
            img, mask = test_ds[idx]
            inp       = img.unsqueeze(0)
            pred      = predict_with_tta(inp, model, device, use_amp).squeeze().cpu().numpy()

            axes[row, 0].imshow(img.squeeze().numpy(), cmap="seismic", vmin=-2, vmax=2, aspect="auto")
            axes[row, 0].set_title(f"Seismic #{idx}", fontsize=9); axes[row, 0].axis("off")

            axes[row, 1].imshow(mask.numpy(), cmap=cmap_facies, vmin=-0.5, vmax=5.5, aspect="auto")
            axes[row, 1].set_title("Ground Truth", fontsize=9); axes[row, 1].axis("off")

            axes[row, 2].imshow(pred, cmap=cmap_facies, vmin=-0.5, vmax=5.5, aspect="auto")
            axes[row, 2].set_title(f"v5 Prediction (TTA)  IoU={metrics['mIoU']:.3f}",
                                    fontsize=9); axes[row, 2].axis("off")

    patches_vis = [mpatches.Patch(color=FACIES_COLORS[j], label=f"S{j}: {CLASS_NAMES[j]}")
                   for j in range(NUM_CLASSES)]
    fig.legend(handles=patches_vis, loc="lower center", ncol=3, fontsize=9,
               bbox_to_anchor=(0.5, -0.01))
    plt.suptitle(f"DeepLabV3+ v5 — Segmentation ({test_name})",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    fname = FIGURES_DIR / f"segmentation_{test_name.replace(' ','_').replace('(','').replace(')','').lower()}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {fname}")

## 13. Confusion Matrix

In [ ]:
for metrics, title in [(metrics1, "Test1_Inline"),
                        (metrics2, "Test2_Crossline"),
                        (metrics_all, "Combined")]:
    cm_arr  = np.array(metrics['confusion_matrix'])
    cm_norm = cm_arr.astype(float) / cm_arr.sum(axis=1, keepdims=True).clip(min=1)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, data, subtitle in zip(axes, [cm_arr, cm_norm], ["Raw Counts", "Row-Normalized"]):
        im = ax.imshow(data, cmap="Blues")
        ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
        ax.set_xticklabels([f"S{j}\n{CLASS_NAMES[j]}" for j in range(NUM_CLASSES)],
                            fontsize=8, rotation=15)
        ax.set_yticklabels([f"S{j} {CLASS_NAMES[j]}" for j in range(NUM_CLASSES)], fontsize=8)
        ax.set_xlabel("Predicted"); ax.set_ylabel("Ground Truth")
        ax.set_title(f"{subtitle}", fontsize=11)
        plt.colorbar(im, ax=ax)
        for i in range(NUM_CLASSES):
            for j in range(NUM_CLASSES):
                val = f"{data[i,j]:.2f}" if subtitle == "Row-Normalized" else f"{int(data[i,j]):,}"
                ax.text(j, i, val, ha="center", va="center", fontsize=6,
                        color="white" if data[i,j] > data.max()*0.5 else "black")

    plt.suptitle(f"DeepLabV3+ v5 — Confusion Matrix ({title})",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    fname = FIGURES_DIR / f"confusion_matrix_{title.lower()}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {fname}")

## 14. Per-class Metrics

In [ ]:
# Combined per-class IoU and Dice
iou_vals  = metrics_all['per_class_iou']
dice_vals = metrics_all['per_class_dice']
x = np.arange(NUM_CLASSES); width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
b1 = ax.bar(x - width/2, iou_vals,  width, label="IoU",  color=FACIES_COLORS,
            alpha=0.85, edgecolor="k", lw=0.5)
b2 = ax.bar(x + width/2, dice_vals, width, label="Dice", color=FACIES_COLORS,
            alpha=0.55, edgecolor="k", lw=0.5, hatch="///")
for bar, v in zip(b1, iou_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{v:.3f}", ha="center", fontsize=8)
for bar, v in zip(b2, dice_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{v:.3f}", ha="center", fontsize=8)
ax.axhline(metrics_all['mIoU'],  color="green",  ls="--", lw=1.2,
           label=f"mIoU={metrics_all['mIoU']:.3f}")
ax.axhline(metrics_all['mean_dice'], color="purple", ls="--", lw=1.2,
           label=f"mDice={metrics_all['mean_dice']:.3f}")
ax.set_xticks(x)
ax.set_xticklabels([f"S{j}\n{CLASS_NAMES[j]}" for j in range(NUM_CLASSES)], fontsize=9)
ax.set_ylabel("Score"); ax.set_ylim(0, 1.1)
ax.set_title("Per-class IoU and Dice — Combined Test Set (v5)", fontsize=11)
ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "per_class_metrics_v5.png", dpi=150, bbox_inches="tight")
plt.show()

# Test1 vs Test2 per-class IoU comparison
fig2, ax2 = plt.subplots(figsize=(12, 5))
t1_iou = metrics1['per_class_iou']
t2_iou = metrics2['per_class_iou']
b3 = ax2.bar(x - width/2, t1_iou,  width, label="Test1 (Inline)",     color="#378ADD",
             alpha=0.85, edgecolor="k", lw=0.5)
b4 = ax2.bar(x + width/2, t2_iou,  width, label="Test2 (Crossline)",  color="#D85A30",
             alpha=0.85, edgecolor="k", lw=0.5)
for bar, v in zip(b3, t1_iou):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f"{v:.3f}", ha="center", fontsize=8)
for bar, v in zip(b4, t2_iou):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f"{v:.3f}", ha="center", fontsize=8)
ax2.set_xticks(x)
ax2.set_xticklabels([f"S{j}\n{CLASS_NAMES[j]}" for j in range(NUM_CLASSES)], fontsize=9)
ax2.set_ylabel("IoU"); ax2.set_ylim(0, 1.1)
ax2.set_title("Per-class IoU: Test1 (Inline) vs Test2 (Crossline) — v5", fontsize=11)
ax2.legend(fontsize=10); ax2.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "per_class_t1_vs_t2_v5.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {FIGURES_DIR}")

## 15. Final Summary

In [ ]:
print("=" * 65)
print(" DEEPLABV3+ v5 — FINAL RESULTS")
print("=" * 65)
print(f"  Pixel Accuracy (PA)  : {metrics_all['PA']*100:.2f}%")
print(f"  Mean Class Acc (MCA) : {metrics_all['MCA']*100:.2f}%")
print(f"  Mean IoU (mIoU)      : {metrics_all['mIoU']*100:.2f}%")
print(f"  Mean Dice            : {metrics_all['mean_dice']*100:.2f}%")
print()
print(f"  Test1 mIoU : {metrics1['mIoU']*100:.2f}%  (inline)")
print(f"  Test2 mIoU : {metrics2['mIoU']*100:.2f}%  (crossline — generalization)")
print()
print("  Version history:")
print("    v3: Test1=64.5%  Test2=27.1%  Combined=40.1%  PA=69.5%")
print("    v4: Test1=76.4%  Test2=68.4%  Combined=76.8%  PA=93.5%")
print(f"    v5: Test1={metrics1['mIoU']*100:.1f}%  Test2={metrics2['mIoU']*100:.1f}%  Combined={metrics_all['mIoU']*100:.1f}%  PA={metrics_all['PA']*100:.1f}%")
print()
print("  Alaudah 2019 reference:")
print("    Patch CNN  : PA=90.5%  MCA=81.7%")
print("    Section CNN: PA=92.2%  MCA=83.9%")
print()
print("  v5 improvements over v4:")
print("    [1] EfficientNet-B4 encoder (better accuracy/VRAM)")
print("    [2] ASPP rates (12,24,36) — wider receptive field")
print("    [3] Image size 256 → 320")
print("    [4] S4 crossline oversampling (WeightedRandomSampler, 3x)")
print("    [5] Mixup augmentation (alpha=0.2, 50% probability)")
print("    [6] Label smoothing CE (eps=0.1)")
print()
print(f"  Model     : {BEST_MODEL_PATH}")
print(f"  Metrics   : {METRICS_DIR}/deeplabv3plus_v5_metrics.json")
print(f"  Figures   : {FIGURES_DIR}/")